# Chapter 12: Sensor Models

<a href="../lite/lab/index.html?path=ch12_sensor_models.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

Your LiDAR says the wall is 3.7 meters away. Is it really? Maybe. But 1 in 100 readings is completely wrong: a bird flew past, a mirror reflected the beam, or the sensor just glitched. If your system trusts every reading equally, that 1 in 100 will ruin everything. Sensor modeling is the art of knowing how much to trust each measurement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import norm, t as student_t

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 12.1 Measurement vs State: The Observation Function

A sensor does not observe the state directly. It observes a **function** of the state, corrupted by noise:

$$\mathbf{z} = h(\mathbf{x}) + \mathbf{v}, \quad \mathbf{v} \sim \mathcal{N}(0, R)$$

For a **range and bearing sensor** (like a LiDAR), the measurement function maps a robot pose $(x_r, y_r, \theta_r)$ and a landmark position $(x_l, y_l)$ to a range and bearing:

$$h(\mathbf{x}) = \begin{bmatrix} \sqrt{(x_l - x_r)^2 + (y_l - y_r)^2} \\ \text{atan2}(y_l - y_r,\, x_l - x_r) - \theta_r \end{bmatrix}$$

**Try it:** Move the robot or the landmark and see how the measurement changes.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
robot_x, robot_y, robot_theta = 1.0, 1.0, np.radians(30)   # robot pose
landmark_x, landmark_y       = 4.0, 3.0                     # landmark
sigma_range   = 0.2   # range noise std (m)     (try 0.05, 0.2, 0.5)
sigma_bearing = 0.05  # bearing noise std (rad) (try 0.01, 0.05, 0.15)
n_samples     = 50    # number of noisy measurements to show
# ─────────────────────────────────────────────────────────────────────────────

def measurement_fn(rx, ry, rth, lx, ly):
    """Compute range and bearing from robot to landmark."""
    dx = lx - rx
    dy = ly - ry
    r = np.sqrt(dx**2 + dy**2)
    b = np.arctan2(dy, dx) - rth
    return r, b

true_range, true_bearing = measurement_fn(
    robot_x, robot_y, robot_theta, landmark_x, landmark_y)

np.random.seed(42)
noisy_ranges   = true_range + np.random.normal(0, sigma_range, n_samples)
noisy_bearings = true_bearing + np.random.normal(0, sigma_bearing, n_samples)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: geometry
ax1.plot(robot_x, robot_y, 'o', color='steelblue', ms=12, zorder=5, label='Robot')
ax1.plot(landmark_x, landmark_y, '*', color='tomato', ms=15, zorder=5, label='Landmark')

# Draw heading
hlen = 0.5
ax1.arrow(robot_x, robot_y, hlen*np.cos(robot_theta), hlen*np.sin(robot_theta),
          head_width=0.1, head_length=0.05, fc='steelblue', ec='steelblue')

# Draw true measurement line
ax1.plot([robot_x, landmark_x], [robot_y, landmark_y], 'k--', lw=1.5, alpha=0.5)

# Draw noisy measurement endpoints
for i in range(n_samples):
    angle = noisy_bearings[i] + robot_theta
    ex = robot_x + noisy_ranges[i] * np.cos(angle)
    ey = robot_y + noisy_ranges[i] * np.sin(angle)
    ax1.plot(ex, ey, '.', color='orange', ms=4, alpha=0.5)

ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
ax1.set_title('Measurement Geometry'); ax1.set_aspect('equal')
ax1.legend(fontsize=9)
ax1.set_xlim(-0.5, 6); ax1.set_ylim(-0.5, 5)

# Right: measurement distribution
ax2.scatter(noisy_ranges, np.degrees(noisy_bearings), s=20, alpha=0.5, color='steelblue',
            label='Noisy measurements')
ax2.plot(true_range, np.degrees(true_bearing), 'x', color='tomato', ms=12, mew=3,
         label='True measurement')
ax2.set_xlabel('Range (m)'); ax2.set_ylabel('Bearing (deg)')
ax2.set_title('Measurement Space')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()
print(f'True range:   {true_range:.3f} m')
print(f'True bearing: {np.degrees(true_bearing):.1f}\u00b0')

## 12.2 Range and Bearing: Likelihood in Position Space

The key question in probabilistic robotics is: **given a measurement, which robot positions are consistent with it?**

For a single **range measurement** $z_r$ to a known landmark, the likelihood $p(z_r \mid x, y)$ is high along a **ring** of radius $z_r$ centered at the landmark. For two range measurements to two different landmarks, the consistent positions are at the **intersection** of two rings.

**Try it:** Change the measured ranges and noise level.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
landmark1    = np.array([2.0, 3.0])   # first landmark position
landmark2    = np.array([6.0, 2.0])   # second landmark position
measured_r1  = 3.0    # measured range to landmark 1 (m)  (try 2.0, 3.0, 4.0)
measured_r2  = 3.5    # measured range to landmark 2 (m)  (try 2.5, 3.5, 5.0)
sigma_range  = 0.3    # range measurement noise (m)       (try 0.1, 0.3, 0.8)
# ─────────────────────────────────────────────────────────────────────────────

x = np.linspace(-2, 10, 300)
y = np.linspace(-2, 8, 300)
X, Y = np.meshgrid(x, y)

# Likelihood for range measurement: Gaussian around true range
dist1 = np.sqrt((X - landmark1[0])**2 + (Y - landmark1[1])**2)
dist2 = np.sqrt((X - landmark2[0])**2 + (Y - landmark2[1])**2)

lik1 = norm.pdf(dist1, measured_r1, sigma_range)
lik2 = norm.pdf(dist2, measured_r2, sigma_range)
lik_combined = lik1 * lik2  # independent measurements multiply

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, lik, title in zip(axes, [lik1, lik2, lik_combined],
    [f'p(z\u2081={measured_r1}m | x,y)', f'p(z\u2082={measured_r2}m | x,y)',
     'Combined Likelihood']):
    im = ax.contourf(X, Y, lik, levels=20, cmap='Blues')
    ax.plot(*landmark1, '*', color='tomato', ms=15, zorder=5, label='Landmark 1')
    ax.plot(*landmark2, '*', color='orange', ms=15, zorder=5, label='Landmark 2')
    ax.set_xlabel('Robot x (m)'); ax.set_ylabel('Robot y (m)')
    ax.set_title(title); ax.set_aspect('equal')
    ax.legend(fontsize=8, loc='upper left')

plt.tight_layout(); plt.show()
print(f'One range reading: ring of consistent positions.')
print(f'Two range readings: the intersection narrows it to (usually) two points.')
print(f'Wider sigma_range: thicker rings, more ambiguity.')

**Key insight:** A single range measurement constrains the robot to a ring. Two measurements constrain it to two points (or one, if they intersect cleanly). Adding a bearing measurement eliminates the remaining ambiguity. This is why **multiple sensor readings** are essential for localization.

## 12.3 Vision as a Sensor: Pinhole Camera Model

A camera projects 3D points onto a 2D image plane. The **pinhole camera model** is:

$$\begin{bmatrix} u \\ v \end{bmatrix} = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \end{bmatrix} \begin{bmatrix} X/Z \\ Y/Z \\ 1 \end{bmatrix}$$

where $(X, Y, Z)$ is the 3D point in the camera frame, $(f_x, f_y)$ is the focal length in pixels, and $(c_x, c_y)$ is the principal point. The **reprojection error** is the difference between the observed pixel location and the projected location.

**Try it:** Change the focal length and the 3D point positions.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
fx, fy  = 400, 400    # focal length (pixels)     (try 200, 400, 800)
cx, cy  = 320, 240    # principal point (pixels)
img_w, img_h = 640, 480  # image dimensions
pixel_noise  = 2.0    # measurement noise (pixels) (try 0.5, 2.0, 5.0)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Generate random 3D landmarks in front of the camera
n_points = 20
points_3d = np.random.uniform([-3, -2, 3], [3, 2, 10], size=(n_points, 3))

# Project to image plane
def project(pts_3d, fx, fy, cx, cy):
    u = fx * pts_3d[:, 0] / pts_3d[:, 2] + cx
    v = fy * pts_3d[:, 1] / pts_3d[:, 2] + cy
    return np.column_stack([u, v])

projected = project(points_3d, fx, fy, cx, cy)

# Add noise to get "observed" pixel locations
observed = projected + np.random.normal(0, pixel_noise, projected.shape)

# Reprojection error
reproj_error = np.sqrt(np.sum((observed - projected)**2, axis=1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Image plane view
ax1.set_xlim(0, img_w); ax1.set_ylim(img_h, 0)  # image coords: y down
ax1.add_patch(plt.Rectangle((0, 0), img_w, img_h, fill=False, edgecolor='k', lw=2))
ax1.scatter(projected[:, 0], projected[:, 1], s=60, color='steelblue',
            marker='o', label='True projection', zorder=4)
ax1.scatter(observed[:, 0], observed[:, 1], s=40, color='tomato',
            marker='x', label='Observed (noisy)', zorder=5)
for i in range(n_points):
    ax1.plot([projected[i, 0], observed[i, 0]],
             [projected[i, 1], observed[i, 1]], 'orange', lw=1, alpha=0.6)
ax1.set_xlabel('u (pixels)'); ax1.set_ylabel('v (pixels)')
ax1.set_title(f'Pinhole Camera Projection (f={fx})'); ax1.legend(fontsize=9)

# Reprojection error histogram
ax2.hist(reproj_error, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(reproj_error.mean(), color='tomato', ls='--', lw=2,
            label=f'Mean = {reproj_error.mean():.2f} px')
ax2.set_xlabel('Reprojection Error (pixels)'); ax2.set_ylabel('Count')
ax2.set_title('Reprojection Error Distribution'); ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()
print(f'Mean reprojection error: {reproj_error.mean():.2f} pixels')
print(f'Expected (theory):       {pixel_noise * np.sqrt(2):.2f} pixels  (\u03c3\u221a2 for 2D Gaussian)')

**Visual SLAM connection:** In visual SLAM, we detect features (corners, blobs) in images and match them across frames. The reprojection error is the cost function minimized by bundle adjustment. Smaller pixel noise means tighter constraints on the 3D structure.

## 12.4 The Beam Model: Four Components of LiDAR

The **beam model** (Thrun, Ch. 6) describes the probability of a LiDAR range reading $z$ given an expected range $z^*$. It mixes four components:

1. **$p_{\text{hit}}$:** Gaussian centered at $z^*$ (correct measurement)
2. **$p_{\text{short}}$:** Exponential decay for $z < z^*$ (unexpected obstacle: a person walked in front)
3. **$p_{\text{max}}$:** Point mass at $z_{\max}$ (sensor failure, max range reading)
4. **$p_{\text{rand}}$:** Uniform over $[0, z_{\max}]$ (random noise, phantom readings)

$$p(z \mid z^*) = w_{\text{hit}}\, p_{\text{hit}} + w_{\text{short}}\, p_{\text{short}} + w_{\text{max}}\, p_{\text{max}} + w_{\text{rand}}\, p_{\text{rand}}$$

**Try it:** Adjust the weights and parameters to see how each component shapes the overall likelihood.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
z_expected = 5.0      # expected range to wall (m)    (try 3.0, 5.0, 8.0)
z_max      = 12.0     # maximum sensor range (m)
sigma_hit  = 0.3      # std dev of hit component      (try 0.1, 0.3, 0.8)
lambda_short = 1.0    # decay rate for short readings  (try 0.5, 1.0, 3.0)
w_hit   = 0.70        # weight: correct measurement    (try 0.5, 0.7, 0.9)
w_short = 0.10        # weight: unexpected obstacle    (try 0.05, 0.1, 0.2)
w_max   = 0.10        # weight: max range failure      (try 0.05, 0.1, 0.2)
w_rand  = 0.10        # weight: random noise           (try 0.05, 0.1, 0.2)
# ─────────────────────────────────────────────────────────────────────────────

# Normalize weights
w_total = w_hit + w_short + w_max + w_rand
w_hit, w_short, w_max, w_rand = (
    w_hit/w_total, w_short/w_total, w_max/w_total, w_rand/w_total)

z = np.linspace(0, z_max, 1000)

# Component 1: Gaussian hit
p_hit = norm.pdf(z, z_expected, sigma_hit)
# Truncate and normalize to [0, z_max]
p_hit[z > z_max] = 0
p_hit /= (p_hit.sum() * (z[1] - z[0]))

# Component 2: Exponential short
p_short = np.where(z <= z_expected,
    lambda_short * np.exp(-lambda_short * z), 0.0)
integral_short = p_short.sum() * (z[1] - z[0])
if integral_short > 0:
    p_short /= integral_short

# Component 3: Max range (narrow spike at z_max)
p_max_arr = np.zeros_like(z)
max_idx = np.argmin(np.abs(z - z_max))
spike_width = max(1, int(0.05 / (z[1] - z[0])))
p_max_arr[max(0, max_idx - spike_width):max_idx + spike_width] = 1.0
integral_max = p_max_arr.sum() * (z[1] - z[0])
if integral_max > 0:
    p_max_arr /= integral_max

# Component 4: Uniform random
p_rand_arr = np.ones_like(z) / z_max

# Combined
p_combined = (w_hit * p_hit + w_short * p_short +
              w_max * p_max_arr + w_rand * p_rand_arr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Individual components
ax1.plot(z, w_hit * p_hit, 'steelblue', lw=2, label=f'Hit (w={w_hit:.2f})')
ax1.plot(z, w_short * p_short, 'orange', lw=2, label=f'Short (w={w_short:.2f})')
ax1.plot(z, w_max * p_max_arr, 'forestgreen', lw=2, label=f'Max (w={w_max:.2f})')
ax1.plot(z, w_rand * p_rand_arr, 'tomato', lw=2, ls='--', label=f'Random (w={w_rand:.2f})')
ax1.axvline(z_expected, color='k', ls=':', alpha=0.5, label=f'z* = {z_expected}')
ax1.set_xlabel('Range z (m)'); ax1.set_ylabel('Weighted Density')
ax1.set_title('Four Components of the Beam Model')
ax1.legend(fontsize=9); ax1.set_xlim(0, z_max + 0.5)

# Combined
ax2.fill_between(z, p_combined, alpha=0.3, color='steelblue')
ax2.plot(z, p_combined, 'steelblue', lw=2.5)
ax2.axvline(z_expected, color='tomato', ls='--', lw=2, label=f'z* = {z_expected}')
ax2.set_xlabel('Range z (m)'); ax2.set_ylabel('p(z | z*)')
ax2.set_title('Combined Beam Model Likelihood')
ax2.legend(fontsize=10); ax2.set_xlim(0, z_max + 0.5)

plt.tight_layout(); plt.show()
print(f'Weights (normalized): hit={w_hit:.2f}, short={w_short:.2f}, '
      f'max={w_max:.2f}, rand={w_rand:.2f}')

**Why four components?**
- The **hit** component handles the vast majority of readings: the beam bounces off the expected surface.
- The **short** component accounts for unexpected obstacles closer than the wall.
- The **max** component handles sensor saturation (beam goes past max range).
- The **random** component captures electronic noise, multi path reflections, and other anomalies.

## 12.5 Noise and Outliers: Heavy Tailed Distributions

Gaussian noise is mathematically convenient, but real sensors produce **outliers**: readings that are far from the truth. A Gaussian assigns near zero probability to these, which can cause catastrophic failures in estimation.

**Heavy tailed distributions** like the **Student's t distribution** assign more probability to extreme values:

$$p(x \mid \nu) \propto \left(1 + \frac{x^2}{\nu}\right)^{-(\nu+1)/2}$$

The parameter $\nu$ (degrees of freedom) controls how heavy the tails are. As $\nu \to \infty$, it approaches a Gaussian.

**Try it:** Generate outlier contaminated data and compare fits.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
n_measurements   = 500   # total measurements           (try 200, 500, 2000)
outlier_fraction = 0.05  # fraction of outliers          (try 0.02, 0.05, 0.15)
true_range       = 5.0   # true range to wall (m)
inlier_std       = 0.3   # inlier noise std (m)         (try 0.1, 0.3, 0.5)
outlier_spread   = 5.0   # outlier spread (m)           (try 2.0, 5.0, 10.0)
t_dof            = 3.0   # Student t degrees of freedom (try 2, 3, 5, 30)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)
n_inliers = int(n_measurements * (1 - outlier_fraction))
n_outliers = n_measurements - n_inliers

inliers  = true_range + np.random.normal(0, inlier_std, n_inliers)
outliers = true_range + np.random.uniform(-outlier_spread, outlier_spread, n_outliers)
measurements = np.concatenate([inliers, outliers])
np.random.shuffle(measurements)

# Fit Gaussian
gauss_mu = measurements.mean()
gauss_std = measurements.std()

# Fit Student t (using scipy)
t_params = student_t.fit(measurements)
t_df, t_loc, t_scale = t_params

z_grid = np.linspace(true_range - 6, true_range + 6, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histogram with fits
ax1.hist(measurements, bins=50, density=True, alpha=0.5, color='steelblue',
         edgecolor='white', label='Data')
ax1.plot(z_grid, norm.pdf(z_grid, gauss_mu, gauss_std), 'tomato', lw=2.5,
         label=f'Gaussian (\u03bc={gauss_mu:.2f}, \u03c3={gauss_std:.2f})')
ax1.plot(z_grid, student_t.pdf(z_grid, *t_params), 'forestgreen', lw=2.5,
         label=f'Student t (\u03bd={t_df:.1f}, loc={t_loc:.2f})')
ax1.axvline(true_range, color='k', ls=':', lw=2, label=f'True range = {true_range}')
ax1.set_xlabel('Range (m)'); ax1.set_ylabel('Density')
ax1.set_title('Gaussian vs Student t Fit'); ax1.legend(fontsize=8)

# Log scale to show tails
ax2.hist(measurements, bins=50, density=True, alpha=0.5, color='steelblue',
         edgecolor='white', label='Data')
ax2.plot(z_grid, norm.pdf(z_grid, gauss_mu, gauss_std), 'tomato', lw=2.5,
         label='Gaussian')
ax2.plot(z_grid, student_t.pdf(z_grid, *t_params), 'forestgreen', lw=2.5,
         label='Student t')
ax2.set_yscale('log'); ax2.set_ylim(1e-4, None)
ax2.set_xlabel('Range (m)'); ax2.set_ylabel('Density (log scale)')
ax2.set_title('Log Scale: Tail Behavior'); ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

# Compute log likelihoods
ll_gauss = np.sum(norm.logpdf(measurements, gauss_mu, gauss_std))
ll_t     = np.sum(student_t.logpdf(measurements, *t_params))
print(f'Gaussian  mean estimate: {gauss_mu:.3f}  (true: {true_range})')
print(f'Student t loc  estimate: {t_loc:.3f}  (true: {true_range})')
print(f'\nLog likelihood  Gaussian: {ll_gauss:.1f}')
print(f'Log likelihood Student t: {ll_t:.1f}')
print(f'Student t fits better by {ll_t - ll_gauss:.1f} log units')

**Key insight:** The Gaussian mean is pulled toward outliers, while the Student t location parameter stays closer to the true value. On the log scale plot, you can see that the Gaussian tails drop off much faster than the data; the Student t tails follow the data more faithfully. In robust SLAM systems, using heavy tailed distributions (or explicit outlier rejection) is essential.

## 12.6 Forward vs Inverse Sensor Models

There are two ways to think about a sensor model:

- **Forward model** $p(z \mid x)$: Given a robot position, what measurements do I expect? This is what we have been building.
- **Inverse model** $p(x \mid z)$: Given a measurement, where is the robot? This is what we want for localization.

Bayes' rule connects them:

$$p(x \mid z) = \frac{p(z \mid x)\, p(x)}{p(z)}$$

Most filters (Kalman, particle) use the **forward model** and apply Bayes' rule. Occupancy grid mapping uses the **inverse model** directly.

**Try it:** Compare the forward and inverse views for a range sensor.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
landmark_pos   = np.array([5.0, 4.0])  # landmark position
measured_range = 3.0    # observed range (m)       (try 2.0, 3.0, 5.0)
sigma_r        = 0.4    # range noise std (m)      (try 0.1, 0.4, 1.0)
prior_center   = np.array([3.0, 2.0])  # prior mean for robot position
prior_std      = 2.0    # prior uncertainty (m)    (try 1.0, 2.0, 4.0)
# ─────────────────────────────────────────────────────────────────────────────

x = np.linspace(-2, 10, 300)
y = np.linspace(-2, 10, 300)
X, Y = np.meshgrid(x, y)

# Forward model: p(z | x, y) for the observed range
dist_to_landmark = np.sqrt((X - landmark_pos[0])**2 + (Y - landmark_pos[1])**2)
forward = norm.pdf(dist_to_landmark, measured_range, sigma_r)

# Prior: Gaussian over robot position
prior = np.exp(-0.5 * ((X - prior_center[0])**2 + (Y - prior_center[1])**2) / prior_std**2)
prior /= prior.sum() * (x[1] - x[0]) * (y[1] - y[0])

# Posterior (inverse model via Bayes): p(x|z) \u221d p(z|x) p(x)
posterior = forward * prior
posterior /= posterior.sum() * (x[1] - x[0]) * (y[1] - y[0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['Forward: p(z | x)', 'Prior: p(x)', 'Posterior: p(x | z)']
data = [forward, prior, posterior]
cmaps = ['Blues', 'Oranges', 'Greens']

for ax, d, title, cmap in zip(axes, data, titles, cmaps):
    ax.contourf(X, Y, d, levels=20, cmap=cmap)
    ax.plot(*landmark_pos, '*', color='tomato', ms=15, zorder=5)
    ax.annotate('Landmark', landmark_pos, fontsize=9,
                xytext=(10, 10), textcoords='offset points')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_title(title); ax.set_aspect('equal')

# Mark prior center
axes[1].plot(*prior_center, 'o', color='steelblue', ms=10, zorder=5)
axes[2].plot(*prior_center, 'o', color='steelblue', ms=10, zorder=5, label='Prior mean')
axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()
print(f'Forward model: ring of positions consistent with range = {measured_range} m')
print(f'Prior: our belief about the robot position before the measurement')
print(f'Posterior: the intersection of the ring and the prior (Bayes update)')

**The Bayesian insight:** The forward model alone gives a ring of possible positions. The prior concentrates probability near where we think the robot is. The posterior (their product, normalized) is concentrated at the intersection: positions that are both likely under the prior and consistent with the measurement. This is the heart of every probabilistic SLAM algorithm.

## 12.7 Capstone: Full 2D LiDAR Beam Model

Let us build a complete beam model for a 2D LiDAR in a rectangular room. The robot takes a **360 degree scan** (one range reading per degree) and we compare:
1. The **ideal** scan (no noise)
2. The **noisy** scan (Gaussian noise on each beam)
3. The **outlier contaminated** scan (beam model with short, max, and random components)

We then compute the **log likelihood** of the scan at the true position vs a wrong position, showing how the sensor model discriminates between hypotheses.

**Try it:** Move the robot, change the noise, or resize the room.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
room_w, room_h = 10.0, 8.0     # room dimensions (m)
robot_x_pos    = 4.0            # robot x position (m)      (try 2, 4, 7)
robot_y_pos    = 3.0            # robot y position (m)      (try 2, 3, 6)
wrong_x, wrong_y = 6.0, 5.0    # wrong position hypothesis
n_beams        = 360            # number of beams (one per degree)
max_range      = 15.0           # max sensor range (m)
hit_sigma      = 0.15           # Gaussian noise std (m)    (try 0.05, 0.15, 0.4)
outlier_rate   = 0.05           # fraction of outlier beams (try 0.02, 0.05, 0.15)
# ─────────────────────────────────────────────────────────────────────────────

def ray_cast_room(rx, ry, angle, room_w, room_h, max_range):
    """Cast a ray from (rx, ry) at given angle. Return distance to nearest wall."""
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    
    distances = []
    # Right wall (x = room_w)
    if cos_a > 1e-9:
        t = (room_w - rx) / cos_a
        y_hit = ry + t * sin_a
        if 0 <= y_hit <= room_h and t > 0:
            distances.append(t)
    # Left wall (x = 0)
    if cos_a < -1e-9:
        t = -rx / cos_a
        y_hit = ry + t * sin_a
        if 0 <= y_hit <= room_h and t > 0:
            distances.append(t)
    # Top wall (y = room_h)
    if sin_a > 1e-9:
        t = (room_h - ry) / sin_a
        x_hit = rx + t * cos_a
        if 0 <= x_hit <= room_w and t > 0:
            distances.append(t)
    # Bottom wall (y = 0)
    if sin_a < -1e-9:
        t = -ry / sin_a
        x_hit = rx + t * cos_a
        if 0 <= x_hit <= room_w and t > 0:
            distances.append(t)
    
    if distances:
        return min(min(distances), max_range)
    return max_range

# Generate scans
angles = np.linspace(0, 2 * np.pi, n_beams, endpoint=False)

# Ideal scan from true position
ideal_scan = np.array([ray_cast_room(robot_x_pos, robot_y_pos, a, room_w, room_h, max_range)
                       for a in angles])

# Ideal scan from wrong position (for likelihood comparison)
wrong_scan = np.array([ray_cast_room(wrong_x, wrong_y, a, room_w, room_h, max_range)
                       for a in angles])

np.random.seed(42)
# Noisy scan (Gaussian only)
noisy_scan = ideal_scan + np.random.normal(0, hit_sigma, n_beams)
noisy_scan = np.clip(noisy_scan, 0, max_range)

# Outlier contaminated scan
contaminated_scan = noisy_scan.copy()
n_outliers = int(n_beams * outlier_rate)
outlier_idx = np.random.choice(n_beams, n_outliers, replace=False)
for idx in outlier_idx:
    rtype = np.random.choice(['short', 'max', 'random'])
    if rtype == 'short':
        contaminated_scan[idx] = np.random.uniform(0, ideal_scan[idx] * 0.5)
    elif rtype == 'max':
        contaminated_scan[idx] = max_range
    else:
        contaminated_scan[idx] = np.random.uniform(0, max_range)

# Convert to Cartesian for plotting
def scan_to_xy(rx, ry, ranges, angles):
    x = rx + ranges * np.cos(angles)
    y = ry + ranges * np.sin(angles)
    return x, y

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
scan_data = [
    (ideal_scan, 'Ideal Scan', 'steelblue'),
    (noisy_scan, f'Noisy (\u03c3={hit_sigma})', 'orange'),
    (contaminated_scan, f'Contaminated ({outlier_rate*100:.0f}% outliers)', 'tomato')
]

for ax, (scan, title, color) in zip(axes, scan_data):
    # Draw room
    room = plt.Rectangle((0, 0), room_w, room_h, fill=False, edgecolor='k', lw=2)
    ax.add_patch(room)
    
    # Draw scan points
    sx, sy = scan_to_xy(robot_x_pos, robot_y_pos, scan, angles)
    ax.scatter(sx, sy, s=3, color=color, alpha=0.6)
    ax.plot(robot_x_pos, robot_y_pos, 'o', color='forestgreen', ms=8, zorder=5)
    
    ax.set_xlim(-1, room_w + 1); ax.set_ylim(-1, room_h + 1)
    ax.set_aspect('equal'); ax.set_title(title)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

plt.tight_layout(); plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────────
# (uses variables from the cell above)
beam_sigma   = 0.3    # sigma for beam model likelihood  (try 0.1, 0.3, 0.6)
# ─────────────────────────────────────────────────────────────────────────────

# Compute log likelihood of the noisy scan at the true vs wrong position
def beam_log_likelihood(observed_scan, expected_ranges, sigma, z_max):
    """Compute log likelihood using a simple Gaussian beam model."""
    ll = 0.0
    for z_obs, z_exp in zip(observed_scan, expected_ranges):
        # Simple model: Gaussian + small uniform (for robustness)
        p_hit = norm.pdf(z_obs, z_exp, sigma)
        p_rand = 1.0 / z_max
        p = 0.95 * p_hit + 0.05 * p_rand
        ll += np.log(max(p, 1e-300))
    return ll

ll_true  = beam_log_likelihood(contaminated_scan, ideal_scan, beam_sigma, max_range)
ll_wrong = beam_log_likelihood(contaminated_scan, wrong_scan, beam_sigma, max_range)

# Scan likelihood over a grid of positions
grid_res = 0.25
gx = np.arange(grid_res, room_w - grid_res, grid_res)
gy = np.arange(grid_res, room_h - grid_res, grid_res)
GX, GY = np.meshgrid(gx, gy)
ll_grid = np.zeros_like(GX)

for i in range(GX.shape[0]):
    for j in range(GX.shape[1]):
        expected = np.array([ray_cast_room(GX[i, j], GY[i, j], a, room_w, room_h, max_range)
                            for a in angles])
        ll_grid[i, j] = beam_log_likelihood(contaminated_scan, expected, beam_sigma, max_range)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Log likelihood map
im = ax1.contourf(GX, GY, ll_grid, levels=30, cmap='viridis')
ax1.plot(robot_x_pos, robot_y_pos, 'o', color='tomato', ms=12, zorder=5,
         label=f'True pos (LL={ll_true:.0f})')
ax1.plot(wrong_x, wrong_y, 's', color='orange', ms=12, zorder=5,
         label=f'Wrong pos (LL={ll_wrong:.0f})')
room = plt.Rectangle((0, 0), room_w, room_h, fill=False, edgecolor='white', lw=2)
ax1.add_patch(room)
ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
ax1.set_title('Log Likelihood Map of Scan'); ax1.set_aspect('equal')
ax1.legend(fontsize=9, loc='lower right')
plt.colorbar(im, ax=ax1, label='Log Likelihood')

# Per beam comparison
beam_indices = np.arange(n_beams)
ax2.plot(beam_indices, contaminated_scan, 'steelblue', lw=0.8, alpha=0.7, label='Observed')
ax2.plot(beam_indices, ideal_scan, 'forestgreen', lw=1.5, label='Expected (true pos)')
ax2.plot(beam_indices, wrong_scan, 'tomato', lw=1.5, alpha=0.5, label='Expected (wrong pos)')
ax2.set_xlabel('Beam Index (degrees)'); ax2.set_ylabel('Range (m)')
ax2.set_title('Scan Comparison: True vs Wrong Position')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'Log likelihood at TRUE  position ({robot_x_pos}, {robot_y_pos}): {ll_true:.1f}')
print(f'Log likelihood at WRONG position ({wrong_x}, {wrong_y}): {ll_wrong:.1f}')
print(f'Difference: {ll_true - ll_wrong:.1f} log units')
print(f'\nThe true position has MUCH higher likelihood.')
print(f'This is how particle filters and scan matching decide where the robot is.')

**What to observe:**
- The log likelihood map has a clear peak at the true robot position.
- The wrong position has a much lower likelihood because its expected scan does not match the observed one.
- Outlier contaminated beams lower the overall likelihood, but if the beam model accounts for them (via the random component), the true position still wins.
- The beam by beam comparison shows where the scans agree and disagree; large deviations correspond to outliers or wrong positions.

## Exercises

**Exercise 12.1:** Add a second landmark to the range and bearing visualization (Section 12.1). Compute measurements to both landmarks and show how two measurements constrain the robot pose more tightly than one.

In [ ]:
# Hint: Compute range and bearing to two landmarks.
# Draw both measurement lines and the intersection region.
# Your code here

**Exercise 12.2:** Implement the **inverse sensor model** for an occupancy grid. Given a single LiDAR beam of range $z$ at angle $\theta$, assign log odds to cells along the beam: free cells get a negative update, the hit cell gets a positive update, and cells beyond the hit are unchanged.

In [ ]:
# Hint: Create a grid of cells. For each cell along the beam,
# compute the distance from the robot and update the log odds.
# Your code here

**Exercise 12.3:** Fit the four beam model parameters ($w_{\text{hit}}, w_{\text{short}}, w_{\text{max}}, w_{\text{rand}}$, plus $\sigma_{\text{hit}}$ and $\lambda_{\text{short}}$) to simulated data using maximum likelihood. Generate 1000 range readings from a known beam model, then recover the parameters.

In [ ]:
# Hint: Use scipy.optimize.minimize to maximize the log likelihood
# over the beam model parameters.
# Your code here

**Exercise 12.4 (challenge):** Implement a **correlation based scan matcher**. Given two LiDAR scans from slightly different positions, search over a grid of $(\Delta x, \Delta y, \Delta \theta)$ to find the transformation that maximizes the overlap between them. Visualize the correlation surface.

In [ ]:
# Hint: Convert scans to point clouds, discretize into a grid,
# and compute the cross correlation for each candidate offset.
# Your code here